# 02 - Data Cleaning

**Goal:** Prepare raw data for analysis.
**Input:** `data/raw/cnc_machining_data.csv`  |  **Output:** `data/processed/cnc_clean.csv`

## Cleaning steps
1. **Duplicates:** 20 duplicate rows removed (2,020 → 2,000 rows).
2. **Missing values:** ~3% missing in `vibration_mm_s` and `coolant_flow_lpm` (simulated sensor failure). Filled using the **median**, which is robust to outliers.
3. **Outliers:** Detected with the IQR method but **not removed**. In machining, extreme vibration or temperature can signal real tool failure.
4. **Sanity checks:** No negative speed, feed, wear or vibration. No invalid defect labels.

 **Note:** Median imputation was applied on the full dataset. In a production pipeline, imputation should be fit on training data only to avoid data leakage.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/raw/cnc_machining_data.csv")
print("Original shape:", df.shape)
df.head()

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("shape after removing dupilcate row", df.shape)

In [ ]:
missing = df.isnull().sum()
missing_pct = (df.isnull().mean()*100).round(2)

pd.DataFrame({"missing_count": missing, "missing_%": missing_pct})

In [ ]:
plt.Figure(figsize=(8,4))
sns.heatmap(df.isnull(),
            cbar=False,
            yticklabels=False,
            cmap="viridis")
plt.title("Missing Values Map (yellow = missing)")
plt.show()

In [ ]:
for col in ["vibration_mm_s", "coolant_flow_lpm"]:
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)
    print(f"{col}: filled with median = {median_value}")

print("\nRemaining missing:", df.isnull().sum().sum())

In [ ]:
num_cols = ["spindle_speed_rpm", "feed_rate_mm_min", "cutting_temp_c",
            "vibration_mm_s", "tool_wear_min", "coolant_flow_lpm"]

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col:22s} outliers: {n_out:4d}  (limits: {lower:.1f} to {upper:.1f})")

In [ ]:
plt.figure(figsize=(14, 7))

for i, col in enumerate(num_cols):
    plt.subplot(2, 3, i + 1)
    sns.boxplot(y=df[col], color="skyblue")
    plt.title(col)

plt.tight_layout()
plt.show()

In [ ]:

checks = {
    "Negative speed":      (df["spindle_speed_rpm"] <= 0).sum(),
    "Negative feed":       (df["feed_rate_mm_min"] <= 0).sum(),
    "Negative tool wear":  (df["tool_wear_min"] < 0).sum(),
    "Negative vibration":  (df["vibration_mm_s"] < 0).sum(),
    "Temp > 500 C":        (df["cutting_temp_c"] > 500).sum(),
    "Invalid defect label":(~df["defect"].isin([0, 1])).sum(),
}
for k, v in checks.items():
    print(f"{k:20s}: {v}")

In [ ]:
df.to_csv("../data/processed/cnc_clean.csv", index=False)
print("Clean data saved ✅  Shape:", df.shape)